# Impact of Two Fluid Spheres (2D)

Two circular bodies of free-surface fluid, given equal and opposite velocities,
meeting head-on in a periodic box. There is no boundary, no gravity and no
physical viscosity: everything that happens after contact -- the jet thrown out
sideways, the ring of surface waves, the density spike travelling back into each
body -- comes from the pressure the collision builds and from the free surface
being free.

This is the `impact` case with `--shape circle`, and its sibling
`impact_squares.ipynb` is the same case with `--shape box` colliding along the
other axis. They are two points of a family the case covers as parameters:
any of seventeen shapes, two bodies or a ring of them, head-on or glancing,
each body spinning or not. The last cell of this notebook draws the shapes;
the table below is the whole parameter set.

![](outputs/01-impact_spheres.gif)


## Every knob, and what it does

The parameters cell below is the whole command line of `impact_spheres.py` written out:
`CaseSpec` fields first, then `impactCase.params` -- the case's own physics
knobs, each of which is also a `--flag`. Anything not named there keeps the
value in `impactCase.defaults`/`.params`.

**Discretisation and time stepping** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `256` | particles across the domain; the spacing is `dx = L / nx` |
| `dim` | `2` | this case is 2D only -- the shapes are 2D SDFs |
| `L` | `4.0` | side of the (periodic) box the bodies live in |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | the spheres notebook used symplectic Euler; RK2 is the shared default and works for both |
| `tLimit` | `10.0` | simulated end time; the loop below runs `tLimit / dt` steps |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed, see below |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `shape` | `'circle'` | body shape: any key of `SHAPE_PRESETS` -- `circle`, `box`, `roundedBox`, `rhombus`, `trapezoid`, `parallelogram`, `equilateralTriangle`, `triangleIsosceles`, `pentagon`, `hexagon`, `octogon`, `hexagram`, `star5`, `vesica`, `cutDisk`, `unevenCapsule`, `moon`. The last cell draws all of them. |
| `size` | `0.5` | characteristic half-size of one body |
| `aspectRatio` | `1.0` | squashes the shape in its second direction: `box` becomes a rectangle `size` by `size * aspectRatio` |
| `rotation` | `0.0` | degrees counter-clockwise, each body turned about its own centre |
| `arrangement` | `'pair'` | `'pair'`: two bodies mirrored across the origin. `'ring'`: `nBodies` of them on a circle, all moving inwards |
| `nBodies`, `ringPhase` | `2`, `0.0` | ring only: how many, and the angle (degrees) of the first one |
| `impactAxis` | `0` | pair only: `0` collides along x, `1` along y |
| `separation` | `0.75` | distance from the origin to each body's centre, snapped to the particle lattice |
| `touching`, `gap` | `False`, `1.0` | ignore `separation` and start the bodies `gap` particle spacings apart instead -- measured from the shape, so it holds for any shape and rotation |
| `lateralOffset` | `0.0` | pair only: slides the two bodies sideways in opposite directions, so they hit off-centre and the collision carries angular momentum |
| `impactVelocity` | `0.5` | speed of each body, directed at the origin |
| `impactAngle` | `0.0` | degrees, turning both velocity vectors off the line joining the bodies: `0` is head-on, larger is a glancing blow |
| `spin` | `0.0` | rad/s of solid-body rotation added to each body, opposite signs on a pair (like two gears), taken about the body's own centre of mass |
| `bodies` | `[]` | the escape hatch: an explicit list of body dicts (`shape`, `size`, `aspect`, `rotation`, `args`, `direction`, `offset`, `velocity`, `spin`), one per body, ignoring every arrangement knob above. A list has no command-line form, so this is notebook/`--config` only. |
| `freeSurface` | `True` | surface detection on -- these bodies have a free surface, unlike the periodic cases in this family |
| `rho0`, `inviscid`, `nu` | `1.0`, `True`, `0.0` | rest density and the physical viscosity (off: the only dissipation is the scheme's) |
| `targetDt` | `0.0005` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit, see the IC cell |
| `band` | `0` | particle layers of boundary padding around the domain; none here, the box is periodic |
| `markerSize` | `4` | plot only: particle marker size |

**Three things this family does differently from the compressible notebooks**
(they will bite if `../../compressible/08-hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `impactCase.initialConditions(ctx, system)`.
   That is where `setupWeaklyCompressibleTimestep` picks the sound speed and
   `config.dt` *together* from `targetDt` -- weakly compressible SPH is free to
   choose its own stiffness, so the timestep is fixed first and `c0` follows
   from the acoustic CFL. Skip it and `config.dt` stays `None`.
2. **The loop is `range(nSteps)`.** No case in this family has a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `IMPACT_FIELDS`
   directly rather than `impactCase.setupPlot`/`updatePlot`, which go through
   `openWindow`/`pumpEvents` and do not live-update inside a Jupyter cell in
   this environment -- `08-hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.impact import impactCase, IMPACT_FIELDS
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.cases.weaklyCompressible import SHAPE_PRESETS, shapeArgs, shapeSdf
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `impact_spheres.py`, made explicit and editable here -- the table in the
# intro cell says what each one does. `impactCase.defaults`/`.params`
# are the same values the CLI script starts from.
spec = CaseSpec(caseName=impactCase.name, scheme=impactCase.scheme,
                params=dict(impactCase.params)) \
    .merged(**impactCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=256,
    dim=2,
    L=4.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,
    # No `dt` here: `initialConditions` sets it from `targetDt` below,
    # together with the sound speed.

    # --- output --------------------------------------------------------------
    caseName='01-impactSpheres',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- impact's own knobs ---------------------------------------------------
    params=dict(
        # the bodies
        shape='circle', size=0.5, aspectRatio=1.0, rotation=0.0,
        # where they start
        arrangement='pair', impactAxis=0, separation=0.75,
        touching=False, gap=1.0, lateralOffset=0.0,
        # how they move
        impactVelocity=0.5, impactAngle=0.0, spin=0.0,
        # the fluid
        freeSurface=True, rho0=1.0, targetDt=0.0005, inviscid=True,
        markerSize=4,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`impactCase.buildSystem` -> the SDF per body, `SHAPE_PRESETS` for its
# arguments, one fluid region each), not re-derived here.
#
# `initialConditions` is the call the compressible notebooks do not have: the
# bodies already carry their velocities (each region was built with a
# `velocities` initial condition, which is how a body is told apart from its
# neighbour without testing the sign of a coordinate), and this is where the
# sound speed and `config.dt` are chosen together from `targetDt`.
ctx = buildContext(impactCase, spec)
impactCase.configureScheme(ctx)
system = impactCase.buildSystem(ctx)
impactCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

print(f'dt = {float(ctx.config.dt):.3e}, c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')

In [ ]:
# What was actually built. `ctx.scratch['bodies']` is the resolved body list --
# the arrangement knobs above turned into one dict per body, with the centre
# snapped to the particle lattice and the shape's measured half-extent filled
# in. Hand this list back as `params=dict(bodies=[...])` to place bodies by
# hand instead.
for body in ctx.scratch['bodies']:
    print({k: (v if not isinstance(v, list) else [round(x, 4) for x in v])
           for k, v in body.items() if k != 'args'})

figure, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
domain = ctx.config.domain
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["bodies"])} bodies, '
                     f'separation {ctx.scratch["separation"]:.4f}')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(IMPACT_FIELDS), not impactCase.setupPlot -- see the
# intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, IMPACT_FIELDS)

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = impactCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=impactCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = impactCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, IMPACT_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=impactCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Did it stay weakly compressible?

Below, the density bounds over the run against the +/-1% band the scheme is
supposed to hold, and the kinetic energy (which should fall: the collision
converts it to pressure work and the scheme's dissipation keeps the rest).

In [ ]:
# The one number worth reading off this run: weakly compressible SPH rests on
# the fluid staying within about a percent of `rho0`, and a free-surface impact
# is where that is hardest. `weaklyCompressibleDiagnostics` records the bounds
# every step, so plot them rather than eyeballing the density panel.
figure, axis = plt.subplots(1, 2, figsize=(11, 3.5))
t = [row['t'] for row in trajectory]
axis[0].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[0].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[0].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[0].set_xlabel('t'); axis[0].set_ylabel(r'$\rho$'); axis[0].legend()
axis[1].plot(t, [row['kineticEnergy'] for row in trajectory])
axis[1].set_xlabel('t'); axis[1].set_ylabel('kinetic energy')
figure.tight_layout()

## The other shapes

`--shape` takes any of these. `SHAPE_PRESETS` maps one characteristic `size`
and one `aspectRatio` onto each primitive's own arguments, and the placement
(centring, and `touching`'s "one spacing apart") is *measured* from the shape
rather than tabulated per primitive -- so a rotated triangle or a `moon` is
placed as accurately as a circle. Change `shape=` in the parameters cell and
re-run; nothing else needs to change.

In [ ]:
# Every shape `--shape` accepts, at this notebook's `size`/`aspectRatio` and
# turned 15 degrees to show that the placement is measured rather than assumed.
# `shapeArgs` turns (size, aspect) into that primitive's own argument list, so
# a case never has to know that `sdBox` takes half-extents while `sdHexagon`
# takes a radius.
names = sorted(SHAPE_PRESETS)
n = 200
axes = torch.linspace(-1.0, 1.0, n)
points = torch.stack(torch.meshgrid(axes, axes, indexing='ij'), dim=-1).reshape(-1, 2)

figure, panels = plt.subplots(3, 6, figsize=(14, 7.5))
for panel, name in zip(panels.flatten(), names):
    sdf = shapeSdf(name, args=shapeArgs(name, spec.param('size'), spec.param('aspectRatio')),
                   rotation=15.0)
    distance = sdf(points)[0].detach().reshape(n, n).T
    panel.contourf(axes, axes, (distance < 0).float(), levels=[0.5, 1.5], colors=['#4878d0'])
    panel.contour(axes, axes, distance, levels=[0.0], colors=['black'], linewidths=0.8)
    panel.set_title(name, fontsize=9)
for panel in panels.flatten():
    panel.set_aspect('equal')
    panel.set_xticks([]); panel.set_yticks([])
for panel in panels.flatten()[len(names):]:
    panel.set_visible(False)
figure.tight_layout()